In [223]:
import sys
print(sys.executable)  # Should show path inside your /venv/ folder

c:\Users\ASUS\Documents\Semester6\Big Data\Lazada Scrape\venv\Scripts\python.exe


In [224]:
%pip install undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.


In [225]:
import time
import random
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [226]:
import undetected_chromedriver as uc

def create_driver():
    options = uc.ChromeOptions()
    options.add_argument("--start-maximized")
    
    # Force it to use version 147 to match your browser
    driver = uc.Chrome(options=options, version_main=147) 
    return driver

In [227]:
import os
import shutil

# This finds the hidden .wdm folder where the drivers are saved
cache_dir = os.path.join(os.path.expanduser("~"), ".wdm")

if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("✅ Cache cleared! Run your scraper cell again.")
else:
    print("No cache found.")

No cache found.


In [228]:
PRODUCTS = {
    "Kemeja": [
        "https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html"
        
    ]
}

In [229]:
def scrape_product(driver, url, category, max_reviews=300):
    driver.get(url)
    time.sleep(random.uniform(5, 8))

    print("📜 Scrolling to load reviews...")
    # 1. DEEPER SCROLLING: Scroll more times to ensure lazy-loaded reviews appear
    for _ in range(8): 
        driver.execute_script("window.scrollBy(0, 900);")
        time.sleep(random.uniform(1.5, 2.5))

    reviews = []
    seen = set()
    wait = WebDriverWait(driver, 15)

    try:
        # 2. BROADER SEARCH: Check for multiple possible review container names
        try:
            wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "#module_product_review, div.pdp-mod-review-main")
                )
            )
        except:
            # 3. DEBUG SCREENSHOT: If it fails, take a picture so we can see why
            print("⚠️ Timeout: Could not find the review section!")
            driver.save_screenshot("lazada_error.png")
            print("📸 Saved 'lazada_error.png'. Please open this file to see what blocked the bot.")
            return reviews # Exit cleanly without crashing

        while len(reviews) < max_reviews:

            items = driver.find_elements(
                By.CSS_SELECTOR, "div.pdp-mod-review-main div.item, div.mod-reviews div.item"
            )

            if not items:
                break

            last_first = items[0].text

            for item in items:
                if len(reviews) >= max_reviews:
                    break

                try:
                    text = item.find_element(
                        By.CSS_SELECTOR,
                        "div.item-content-main-content-reviews span, div.content"
                    ).text.strip()

                    if text == "":
                        text = None
                except:
                    text = None

                rating = 0
                try:
                    stars = item.find_elements(
                        By.CSS_SELECTOR,
                        ".i-rate-star-item svg, img.star"
                    )
                    
                    # Lazada sometimes uses images, sometimes SVGs for stars
                    for s in stars:
                        if s.tag_name == "svg":
                            paths = s.find_elements(By.TAG_NAME, "path")
                            if len(paths) > 1:
                                style = paths[1].get_attribute("style") or ""
                                if "255, 200, 60" in style or "orange" in style.lower():
                                    rating += 1
                        elif s.tag_name == "img":
                            src = s.get_attribute("src") or ""
                            if "TB19ZvEgfAWYK20" in src or "star-yellow" in src: # common lazada yellow star image
                                rating += 1
                except:
                    rating = 0

                key = (text, rating)

                if key not in seen:
                    seen.add(key)

                    reviews.append({
                        "category": category,
                        "product_url": url,
                        "review": text if text is not None else np.nan,
                        "rating": rating
                    })

            print(f"📦 {url} → collected: {len(reviews)}/{max_reviews}")

            clicked = False

            # Load More / Next Page Logic
            try:
                next_btn = driver.find_element(
                    By.XPATH,
                    "//li[contains(@class,'next')]//button | //button[contains(., 'Next')]"
                )

                if "disabled" not in next_btn.get_attribute("class"):
                    # Scroll slightly up so the button isn't hidden by bottom banners
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                    clicked = True
                else:
                    break
            except:
                break

            if not clicked:
                break

            new_items = driver.find_elements(
                By.CSS_SELECTOR,
                "div.pdp-mod-review-main div.item, div.mod-reviews div.item"
            )

            if new_items and new_items[0].text == last_first:
                break

    except Exception as e:
        print("❌ General Error:", e)

    return reviews

In [230]:
def run_pipeline():
    driver = create_driver()

    # --- UPDATED: Set the exact limit to 250 ---
    CATEGORY_LIMIT = 250
    GLOBAL_LIMIT = 250

    final_data = []
    category_count = {c: 0 for c in PRODUCTS}

    try:
        for category, urls in PRODUCTS.items():

            print(f"\n🚀 CATEGORY START: {category}")

            for url in urls:

                if len(final_data) >= GLOBAL_LIMIT:
                    print("✅ Global limit reached (250)")
                    return final_data

                if category_count[category] >= CATEGORY_LIMIT:
                    print(f"✅ Category {category} done (250)")
                    break

                remaining = CATEGORY_LIMIT - category_count[category]
                fetch_limit = min(250, remaining) # Passes exactly 250 to the scraper

                print(f"⏳ Scraping {url}")

                data = scrape_product(driver, url, category, fetch_limit)

                final_data.extend(data)
                category_count[category] += len(data)

                print(f"📦 {category}: {category_count[category]} collected")

                time.sleep(random.uniform(3, 6))
                
                # --- UPDATED: Stop entirely after processing 1 link ---
                print("🛑 Stopping after 1 link as requested.")
                return final_data

    finally:
        driver.quit()

    return final_data

In [231]:
data = run_pipeline()

df = pd.DataFrame(data)
df.to_csv("lazada_kemeja_category_v1.csv", index=False)

print("\n🎉 DONE")
print(df.head())


🚀 CATEGORY START: Kemeja
⏳ Scraping https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html
📜 Scrolling to load reviews...
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 3/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 6/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 9/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 14/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 19/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 24/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 28/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 32/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s16095030454.html → collected: 37/250
📦 https://www.lazada.co.id/products/pdp-i8660990474-s1609